# NullVector Cookbook — End-to-End Acquisition, Visual Enrichment, Tree Construction, and Export

This cookbook demonstrates the current NullVector runtime with the real visual contract:
persisted acquisition assets, attachment-only multimodal enrichment, acquisition-backed tree builds,
optional live text summarization, and edge export.


### Environment

- Uses `903000608.pdf` when present, otherwise falls back to the committed Phase 01 born-digital fixture.
- Uses content/settings-derived acquisition and tree run IDs so reruns are idempotent.
- Uses the NullVector attachment-only multimodal gateway with a deterministic noop provider.
- Live text summarization is optional and only runs when `OPENROUTER_API_KEY` is set.


In [1]:
# environment setup
from pathlib import Path
import hashlib
import json
import os
from typing import Any

REPO_ROOT = Path.cwd()
COOKBOOK_ROOT = REPO_ROOT / "cookbook"
COOKBOOK_ARTIFACT_ROOT = COOKBOOK_ROOT / "artifacts"
COOKBOOK_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_PDF = REPO_ROOT / "903000608.pdf"
FIXTURE_PDF = REPO_ROOT / "fixtures" / "pdfs" / "phase01" / "born_digital_with_outline.pdf"
PDF_PATH = LOCAL_PDF if LOCAL_PDF.exists() else FIXTURE_PDF
PDF_SOURCE_MODE = "local_real_pdf" if LOCAL_PDF.exists() else "fixture_pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"No source PDF available at {LOCAL_PDF} or {FIXTURE_PDF}")

print(json.dumps({
    "repo_root": str(REPO_ROOT),
    "pdf_path": str(PDF_PATH),
    "pdf_source_mode": PDF_SOURCE_MODE,
    "cookbook_artifact_root": str(COOKBOOK_ARTIFACT_ROOT),
}, indent=2))


{
  "repo_root": "/home/pruthvi/projects/NullVector",
  "pdf_path": "/home/pruthvi/projects/NullVector/fixtures/pdfs/phase01/born_digital_with_outline.pdf",
  "pdf_source_mode": "fixture_pdf",
  "cookbook_artifact_root": "/home/pruthvi/projects/NullVector/cookbook/artifacts"
}


In [2]:
# imports
from pydantic import TypeAdapter
from nullvector.domain import (
    AcquisitionRequest,
    AcquisitionRunManifest,
    AcquisitionSettings,
    CanonicalDocumentLedger,
    NodeCard,
    NodeSummary,
    StructuredRegionInsight,
    TreeBuildRequest,
    TreeBuildManifest,
    TreeSettings,
    UnassignedPageSpan,
    UnresolvedRegion,
    VisualArtifact,
    VisualEnrichmentRequest,
    VisualRegionReference,
)
from nullvector.export import ExporterDependencyError, to_langchain_documents
from nullvector.ingest import acquire_document
from nullvector.ingest.acquisition_artifacts import settings_digest as acquisition_settings_digest
from nullvector.ingest.fingerprint import fingerprint_document
from nullvector.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayError,
    GatewayService,
    LiteLLMProviderConfig,
    LiteLLMSDKAdapter,
    MultimodalGatewayConfig,
    MultimodalGatewayError,
    MultimodalProviderConfig,
    MultimodalGatewayService,
    NoopMultimodalProviderAdapter,
    NoopMultimodalResponse,
    VisualEnrichmentService,
)
from nullvector.tree import build_tree

print("✓ NullVector imports succeeded")


✓ NullVector imports succeeded


In [3]:
# configuration
def load_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text(encoding="utf-8"))

def load_typed_json(path: str | Path, adapter: TypeAdapter) -> Any:
    return adapter.validate_json(Path(path).read_text(encoding="utf-8"))

def stable_digest(payload: dict[str, Any]) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")
    ).hexdigest()

def tree_settings_digest(settings: TreeSettings) -> str:
    return stable_digest(settings.model_dump(mode="json"))

NODE_CARD_ADAPTER = TypeAdapter(tuple[NodeCard, ...])
NODE_SUMMARY_ADAPTER = TypeAdapter(tuple[NodeSummary, ...])
UNASSIGNED_SPAN_ADAPTER = TypeAdapter(tuple[UnassignedPageSpan, ...])

def choose_visual_attachment_path(region: VisualRegionReference) -> tuple[str | None, str | None]:
    if region.asset_path:
        return region.asset_path, "asset_path"
    if region.page_render_path:
        return region.page_render_path, "page_render_path"
    return None, None

def block_to_region_reference(document_id: str, page_index: int, block: VisualArtifact | UnresolvedRegion) -> VisualRegionReference:
    region_id = block.visual_id if isinstance(block, VisualArtifact) else block.region_id
    image_ref = block.image_ref if isinstance(block, VisualArtifact) else None
    return VisualRegionReference(
        document_id=document_id,
        page_index=page_index,
        region_id=region_id,
        bbox=block.bbox,
        image_ref=image_ref,
        asset_path=block.asset_path,
        page_render_path=block.page_render_path,
        render_dpi=block.render_dpi,
        coordinate_space=block.coordinate_space,
    )

def collect_enrichable_regions(ledger: CanonicalDocumentLedger) -> list[dict[str, Any]]:
    regions: list[dict[str, Any]] = []
    for page in ledger.pages:
        for block in page.blocks:
            if isinstance(block, VisualArtifact):
                if not block.needs_enrichment:
                    continue
                region = block_to_region_reference(ledger.document_id, page.page_index, block)
                attachment_path, attachment_kind = choose_visual_attachment_path(region)
                regions.append({
                    "page_index": page.page_index,
                    "block_type": "visual_artifact",
                    "region": region,
                    "attachment_path": attachment_path,
                    "attachment_kind": attachment_kind,
                    "kind_hint": block.kind_hint,
                    "reason_code": None,
                })
            elif isinstance(block, UnresolvedRegion):
                region = block_to_region_reference(ledger.document_id, page.page_index, block)
                attachment_path, attachment_kind = choose_visual_attachment_path(region)
                regions.append({
                    "page_index": page.page_index,
                    "block_type": "unresolved_region",
                    "region": region,
                    "attachment_path": attachment_path,
                    "attachment_kind": attachment_kind,
                    "kind_hint": None,
                    "reason_code": block.reason_code,
                })
    return regions

def acquire_visual_ready_document(pdf_path: Path) -> tuple[AcquisitionRunManifest, CanonicalDocumentLedger, list[dict[str, Any]], dict[str, Any]]:
    fingerprint = fingerprint_document(str(pdf_path))
    acquisition_settings = AcquisitionSettings()
    settings_tag = acquisition_settings_digest(acquisition_settings)[:10]
    base_run_id = f"cookbook-acquisition-viz-{fingerprint.sha256[:12]}-{settings_tag}"

    def run_with_id(run_id: str) -> AcquisitionRunManifest:
        return acquire_document(
            AcquisitionRequest(
                source_path=str(pdf_path),
                acquisition_run_id=run_id,
                artifact_root=str(COOKBOOK_ARTIFACT_ROOT / "acquisition_runs"),
                settings=acquisition_settings,
            )
        )

    manifest = run_with_id(base_run_id)
    ledger = CanonicalDocumentLedger.model_validate_json(Path(manifest.ledger_path).read_text(encoding="utf-8"))
    regions = collect_enrichable_regions(ledger)
    enrichable_count = len(regions)
    asset_backed_count = sum(1 for region in regions if region["attachment_path"] is not None)

    if enrichable_count > 0 and asset_backed_count < enrichable_count:
        manifest = run_with_id(f"{base_run_id}-refresh")
        ledger = CanonicalDocumentLedger.model_validate_json(Path(manifest.ledger_path).read_text(encoding="utf-8"))
        regions = collect_enrichable_regions(ledger)
        enrichable_count = len(regions)
        asset_backed_count = sum(1 for region in regions if region["attachment_path"] is not None)

    validation = {
        "acquisition_run_id": manifest.acquisition_run_id,
        "enrichable_region_count": enrichable_count,
        "asset_backed_region_count": asset_backed_count,
        "assets_complete": asset_backed_count == enrichable_count if enrichable_count else True,
    }
    return manifest, ledger, regions, validation

def build_tree_manifest(acquisition_manifest: AcquisitionRunManifest, *, summarize: bool = False, gateway: GatewayService | None = None) -> TreeBuildManifest:
    tree_settings = TreeSettings()
    tree_tag = tree_settings_digest(tree_settings)[:10]
    suffix = "summaries" if summarize else "base"
    tree_run_id = f"cookbook-tree-{acquisition_manifest.source_fingerprint.sha256[:12]}-{tree_tag}-{suffix}"
    return build_tree(
        TreeBuildRequest(
            acquisition_manifest_path=str(Path(acquisition_manifest.artifact_root) / "manifest.json"),
            tree_run_id=tree_run_id,
            summarize=summarize,
            settings=tree_settings,
        ),
        gateway=gateway,
    )

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY","REDACTED_OPENROUTER_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
TEXT_LLM_ENABLED = bool(OPENROUTER_API_KEY)

print(json.dumps({
    "text_llm_enabled": TEXT_LLM_ENABLED,
    "openrouter_base": OPENROUTER_API_BASE,
}, indent=2))


{
  "text_llm_enabled": true,
  "openrouter_base": "https://openrouter.ai/api/v1"
}


In [4]:
# execution
fingerprint = fingerprint_document(str(PDF_PATH))
acquisition_manifest, canonical_ledger, visual_regions, visual_validation = acquire_visual_ready_document(PDF_PATH)
acquisition_manifest_path = Path(acquisition_manifest.artifact_root) / "manifest.json"
selected_outline = load_json(acquisition_manifest.selected_outline_path)
projection = load_json(acquisition_manifest.projection_view_path) if acquisition_manifest.projection_view_path else None

print(json.dumps({
    "document_id": fingerprint.document_id,
    "page_count": acquisition_manifest.page_count,
    "outline_source": acquisition_manifest.selected_outline_source.value,
    "outline_entry_count": len(selected_outline["entries"]),
    "projection_path": acquisition_manifest.projection_view_path,
    "visual_validation": visual_validation,
}, indent=2))


{
  "document_id": "015e3d97fc52d47cf609c750ced10fb5f47ad21f1e1601b28dd55ab1b4a54b52",
  "page_count": 3,
  "outline_source": "pymupdf",
  "outline_entry_count": 3,
  "projection_path": "/home/pruthvi/projects/NullVector/cookbook/artifacts/acquisition_runs/cookbook-acquisition-viz-015e3d97fc52-370e10389b/015e3d97fc52d47cf609c750ced10fb5f47ad21f1e1601b28dd55ab1b4a54b52/projection/tree-synthesis-view.json",
  "visual_validation": {
    "acquisition_run_id": "cookbook-acquisition-viz-015e3d97fc52-370e10389b",
    "enrichable_region_count": 0,
    "asset_backed_region_count": 0,
    "assets_complete": true
  }
}


In [5]:
# execution
total_blocks = sum(len(page.blocks) for page in canonical_ledger.pages)
unresolved_count = sum(
    1 for page in canonical_ledger.pages for block in page.blocks if isinstance(block, UnresolvedRegion)
)
visual_artifact_count = sum(
    1 for page in canonical_ledger.pages for block in page.blocks if isinstance(block, VisualArtifact)
)
asset_backed_regions = [region for region in visual_regions if region["attachment_path"] is not None]
sample_region = asset_backed_regions[0] if asset_backed_regions else None

inspection_summary = {
    "ledger_pages": len(canonical_ledger.pages),
    "total_blocks": total_blocks,
    "visual_artifact_count": visual_artifact_count,
    "unresolved_region_count": unresolved_count,
    "enrichable_visual_region_count": len(visual_regions),
    "asset_backed_visual_region_count": len(asset_backed_regions),
    "representative_region": None if sample_region is None else {
        "page_index": sample_region["page_index"],
        "block_type": sample_region["block_type"],
        "region_id": sample_region["region"].region_id,
        "attachment_kind": sample_region["attachment_kind"],
        "attachment_path": sample_region["attachment_path"],
        "page_render_path": sample_region["region"].page_render_path,
        "render_dpi": sample_region["region"].render_dpi,
        "coordinate_space": sample_region["region"].coordinate_space.value,
        "kind_hint": sample_region["kind_hint"],
        "reason_code": sample_region["reason_code"],
    },
}
print(json.dumps(inspection_summary, indent=2))


{
  "ledger_pages": 3,
  "total_blocks": 6,
  "visual_artifact_count": 0,
  "unresolved_region_count": 0,
  "enrichable_visual_region_count": 0,
  "asset_backed_visual_region_count": 0,
  "representative_region": null
}


In [6]:
# execution
multimodal_audit_root = COOKBOOK_ARTIFACT_ROOT / "multimodal-audit"
multimodal_gateway = MultimodalGatewayService(
    MultimodalGatewayConfig(
        provider=MultimodalProviderConfig(model="cookbook-multimodal-noop"),
        audit_root=str(multimodal_audit_root),
    ),
    provider_adapter=NoopMultimodalProviderAdapter(
        {
            "visual_region_enrichment": NoopMultimodalResponse(
                output_json={
                    "insight": {
                        "summary": "Attachment-backed visual region reviewed through the NullVector multimodal gateway.",
                        "labels": ["asset-backed-region", "attachment-only"],
                        "attributes": {"demo": True},
                        "confidence": 0.5,
                    }
                }
            )
        }
    ),
)
visual_enrichment_service = VisualEnrichmentService(multimodal_gateway)
visual_attachment = None
visual_demo = {
    "status": "skipped_no_regions",
    "reason": "No asset-backed enrichable visual regions were available.",
}

if sample_region is not None:
    try:
        visual_attachment = visual_enrichment_service.enrich(
            VisualEnrichmentRequest(
                request_id=f"cookbook-visual-{sample_region['region'].region_id}",
                region=sample_region["region"],
                prompt="Describe the persisted region attachment for cookbook inspection.",
            )
        )
        visual_demo = {
            "status": "enriched",
            "region_id": sample_region["region"].region_id,
            "page_index": sample_region["page_index"],
            "attachment_kind": sample_region["attachment_kind"],
            "attachment_path": sample_region["attachment_path"],
            "page_render_path": sample_region["region"].page_render_path,
            "provider_identity": visual_attachment.provider_identity,
            "authoritative": visual_attachment.authoritative,
            "summary": visual_attachment.insight.summary,
            "labels": list(visual_attachment.insight.labels),
            "audit_path": visual_attachment.audit_path,
        }
    except MultimodalGatewayError as exc:
        visual_demo = {
            "status": "failure",
            "category": exc.failure.category.value,
            "message": str(exc),
            "audit_path": exc.audit_path,
        }

print(json.dumps(visual_demo, indent=2))


{
  "status": "skipped_no_regions",
  "reason": "No asset-backed enrichable visual regions were available."
}


In [7]:
# execution
tree_manifest = build_tree_manifest(acquisition_manifest)
build_report = load_json(tree_manifest.build_report_path)
verification_report = load_json(tree_manifest.verification_report_path)
unassigned_spans = load_typed_json(tree_manifest.unassigned_spans_path, UNASSIGNED_SPAN_ADAPTER)
node_cards = load_typed_json(tree_manifest.node_cards_path, NODE_CARD_ADAPTER)

print(json.dumps({
    "tree_run_id": tree_manifest.tree_run_id,
    "committed_node_count": tree_manifest.committed_node_count,
    "unassigned_span_count": tree_manifest.unassigned_span_count,
    "verification_status": verification_report["status"],
    "outline_trust_mode": build_report["outline_trust_mode"],
    "committed_titles_preview": [card.title for card in node_cards[:8]],
}, indent=2))


{
  "tree_run_id": "cookbook-tree-015e3d97fc52-42c8480bf3-base",
  "committed_node_count": 2,
  "unassigned_span_count": 0,
  "verification_status": "passed",
  "outline_trust_mode": "outline_primary",
  "committed_titles_preview": [
    "Overview",
    "Appendix"
  ]
}


In [8]:
# execution
llm_gateway = None
summarized_manifest = None
summarization_status = {
    "status": "skipped",
    "reason": "OPENROUTER_API_KEY is not set; live text summarization is optional.",
}

if TEXT_LLM_ENABLED:
    llm_gateway_config = GatewayConfig(
        provider=LiteLLMProviderConfig(
            model="openrouter/google/gemini-3.1-flash-lite-preview",
            api_key_env_var="OPENROUTER_API_KEY",
            api_base_env_var="OPENROUTER_API_BASE",
        ),
        audit=GatewayAuditConfig(
            persist_root=str(COOKBOOK_ARTIFACT_ROOT / "llm-audit"),
        ),
        timeout_seconds=60.0,
    )
    llm_gateway = GatewayService(
        llm_gateway_config,
        provider_adapter=LiteLLMSDKAdapter(),
    )
    try:
        summarized_manifest = build_tree_manifest(
            acquisition_manifest,
            summarize=True,
            gateway=llm_gateway,
        )
        summarization_status = {
            "status": "completed",
            "tree_run_id": summarized_manifest.tree_run_id,
            "node_summaries_path": summarized_manifest.node_summaries_path,
        }
    except GatewayError as exc:
        summarization_status = {
            "status": "gateway_error",
            "category": exc.failure.category.value,
            "message": str(exc),
            "audit_path": exc.audit_path,
        }

print(json.dumps(summarization_status, indent=2))


{
  "status": "completed",
  "tree_run_id": "cookbook-tree-015e3d97fc52-42c8480bf3-summaries",
  "node_summaries_path": "/home/pruthvi/projects/NullVector/cookbook/artifacts/acquisition_runs/cookbook-acquisition-viz-015e3d97fc52-370e10389b/015e3d97fc52d47cf609c750ced10fb5f47ad21f1e1601b28dd55ab1b4a54b52/tree/cookbook-tree-015e3d97fc52-42c8480bf3-summaries/strategy/attempts/01-outline_only/summaries/node-summaries.json"
}


In [9]:
# execution
export_manifest = summarized_manifest or tree_manifest
export_cards = load_typed_json(export_manifest.node_cards_path, NODE_CARD_ADAPTER)
summaries_by_id = {}
if export_manifest.node_summaries_path:
    for summary in load_typed_json(export_manifest.node_summaries_path, NODE_SUMMARY_ADAPTER):
        summaries_by_id[summary.node_id] = summary

export_status = {"status": "skipped", "reason": "langchain_core export not attempted yet"}
try:
    attachments_by_node_id = {}
    if visual_attachment is not None and visual_attachment.node_id is not None:
        attachments_by_node_id[visual_attachment.node_id] = (visual_attachment,)
    docs = to_langchain_documents(
        export_cards,
        summaries_by_id=summaries_by_id,
        attachments_by_node_id=attachments_by_node_id,
    )
    export_status = {
        "status": "completed",
        "document_count": len(docs),
        "preview_titles": [doc.metadata["title"] for doc in docs[:3]],
    }
except ExporterDependencyError as exc:
    export_status = {"status": "dependency_missing", "message": str(exc)}

print(json.dumps(export_status, indent=2))


{
  "status": "completed",
  "document_count": 2,
  "preview_titles": [
    "Overview",
    "Appendix"
  ]
}


In [10]:
# inspect results
multimodal_audit_files = sorted((COOKBOOK_ARTIFACT_ROOT / "multimodal-audit").glob("*.json"))
llm_audit_files = sorted((COOKBOOK_ARTIFACT_ROOT / "llm-audit").glob("*.json")) if (COOKBOOK_ARTIFACT_ROOT / "llm-audit").exists() else []

summary = {
    "pdf": {
        "path": str(PDF_PATH),
        "source_mode": PDF_SOURCE_MODE,
        "document_id": fingerprint.document_id,
        "page_count": fingerprint.page_count,
    },
    "acquisition": {
        "run_id": acquisition_manifest.acquisition_run_id,
        "artifact_root": acquisition_manifest.artifact_root,
        "outline_source": acquisition_manifest.selected_outline_source.value,
        "projection_view_path": acquisition_manifest.projection_view_path,
        "canonical_text_substrate_path": acquisition_manifest.canonical_text_substrate_path,
    },
    "visual": {
        "enrichable_region_count": len(visual_regions),
        "asset_backed_region_count": len(asset_backed_regions),
        "demo": visual_demo,
        "multimodal_audit_count": len(multimodal_audit_files),
        "uses_attachment_paths_only": True,
    },
    "tree": {
        "tree_run_id": tree_manifest.tree_run_id,
        "committed_node_count": tree_manifest.committed_node_count,
        "unassigned_span_count": tree_manifest.unassigned_span_count,
        "verification_status": verification_report["status"],
    },
    "summarization": {
        **summarization_status,
        "llm_audit_count": len(llm_audit_files),
    },
    "export": export_status,
}
print(json.dumps(summary, indent=2))


{
  "pdf": {
    "path": "/home/pruthvi/projects/NullVector/fixtures/pdfs/phase01/born_digital_with_outline.pdf",
    "source_mode": "fixture_pdf",
    "document_id": "015e3d97fc52d47cf609c750ced10fb5f47ad21f1e1601b28dd55ab1b4a54b52",
    "page_count": 3
  },
  "acquisition": {
    "run_id": "cookbook-acquisition-viz-015e3d97fc52-370e10389b",
    "artifact_root": "/home/pruthvi/projects/NullVector/cookbook/artifacts/acquisition_runs/cookbook-acquisition-viz-015e3d97fc52-370e10389b/015e3d97fc52d47cf609c750ced10fb5f47ad21f1e1601b28dd55ab1b4a54b52",
    "outline_source": "pymupdf",
    "projection_view_path": "/home/pruthvi/projects/NullVector/cookbook/artifacts/acquisition_runs/cookbook-acquisition-viz-015e3d97fc52-370e10389b/015e3d97fc52d47cf609c750ced10fb5f47ad21f1e1601b28dd55ab1b4a54b52/projection/tree-synthesis-view.json",
    "canonical_text_substrate_path": "/home/pruthvi/projects/NullVector/cookbook/artifacts/acquisition_runs/cookbook-acquisition-viz-015e3d97fc52-370e10389b/015e3d

### Known Limitations

- Visual enrichment in this cookbook is intentionally attachment-only and deterministic via the noop multimodal adapter.
- Live text summarization is optional and skipped unless `OPENROUTER_API_KEY` is configured.
- If a source document has no enrichable visual regions, the visual demo is skipped honestly instead of fabricating output.
